In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate())

print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/03 10:30:34 WARN Utils: Your hostname, oualid-Latitude-E5570, resolves to a loopback address: 127.0.1.1; using 192.168.51.11 instead (on interface wlp1s0)
26/10/03 10:30:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 10:30:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://localhost:4040


In [4]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .master("local[4]")                            # 1 machine, 4 cores = 4 task slots
    .appName("tp")
    .config("spark.sql.shuffle.partitions", "8")   # 200 is too much for a laptop
    .getOrCreate())

# Customers: 1000 rows
customers = spark.range(1, 1001).select(
    F.col("id").alias("cust_id"),
    F.concat(F.lit("name_"), F.col("id")).alias("name"),
    F.element_at(F.array(F.lit("MA"), F.lit("FR"), F.lit("US"), F.lit("DE")),
                 (F.col("id") % 4 + 1).cast("int")).alias("country"))

# Orders: 1,000,000 rows, and half of them belong to customer 1 (a hot key, for the skew lab)
orders = spark.range(1_000_000).select(
    F.col("id").alias("order_id"),
    F.when(F.rand(42) < 0.5, F.lit(1))
     .otherwise((F.rand(7) * 1000).cast("int") + 1).alias("cust_id"),
    (F.rand(1) * 500).alias("amount"))

# Employees: 5000 rows, 10 departments
employees = spark.range(5000).select(
    F.col("id").alias("emp_id"),
    F.concat(F.lit("emp_"), F.col("id")).alias("name"),
    (F.col("id") % 10).alias("dept_id"),
    (F.rand(3) * 8000 + 2000).cast("int").alias("salary"),
    F.date_add(F.lit("2015-01-01"), (F.rand(5) * 3000).cast("int")).alias("hire_date"))

# Save as Parquet so every lab reads the same data
customers.write.mode("overwrite").parquet("data/customers")
orders.write.mode("overwrite").parquet("data/orders")
employees.write.mode("overwrite").parquet("data/employees")

In [5]:
orders = spark.read.parquet("data/orders")
print(orders.count())
print(orders.rdd.getNumPartitions())

1000000
4
